## `Student Performance Prediction System`

### 📝 Project Quick Summary
* **Objective:** An early-warning Machine Learning engine that analyzes academic and lifestyle habits to predict whether a student will **Pass** or **Fail**.
* **The Pipeline:** Built with an advanced scikit-learn `Pipeline` and `ColumnTransformer` that automates text encoding and data scaling seamlessly while maintaining **zero data leakage**.
* **Model Capability:** Powered by a **Decision Tree Classifier** that natively ingests raw string data and directly outputs clean `"Pass"` or `"Fail"` text labels.
* **Production Ready:** Bundled into a single, compact `joblib` file featuring a type-safe fallback system that ensures crash-free predictions even with faulty user inputs.


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
import kagglehub
import os
import shutil
import chardet
import joblib

c:\Users\SHAH COMPUTERS\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sklearn.preprocessing as skp
import sklearn.metrics as skm
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

In [4]:

# Download latest version
path = kagglehub.dataset_download("amrmaree/student-performance-prediction")

print("Path to dataset files:", path)

df = pd.read_csv(f"{path}/student_performance_dataset.csv")

df.head()

ConnectionError: HTTPSConnectionPool(host='api.kaggle.com', port=443): Max retries exceeded with url: /v1/datasets.DatasetApiService/GetDataset (Caused by NameResolutionError("HTTPSConnection(host='api.kaggle.com', port=443): Failed to resolve 'api.kaggle.com' ([Errno 11001] getaddrinfo failed)"))

In [ ]:

if not os.listdir(path):
        shutil.rmtree(path) # Delete folder if it is empty

with open(f"{path}/student_performance_dataset.csv","rb") as f:
        raw_data = f.read(10000)
        result = chardet.detect(raw_data)
print(result)

In [ ]:
df.info()

In [ ]:
df.describe(include="string")

In [ ]:
df.describe(include="string")

In [ ]:
print(f"Total Duplicate Rows: {df.duplicated().sum()}")

# df[df.duplicated()] # Show duplicate rows
# df[df["Student_ID"]=="S458"]

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
missing_data = df.isnull()
for col in df.columns:
    print(missing_data[col].value_counts())
    print(" ")

In [ ]:
df.hist(figsize=(12,8),edgecolor="black")
plt.tight_layout()
plt.show()

In [ ]:
# HeatMap of Correlation  
numeric=df.select_dtypes(include="number").corr() 
# 1 represents strong proportional relation
# 0 represents Neutral relation, it means changing value of one column does not affect on another column values
# -1 represents strong inversly proportional relation
sns.heatmap(numeric,annot=True,cmap="coolwarm")

In [ ]:
df.dropna(subset=["Pass_Fail"],inplace=True)
df.reset_index(drop=True,inplace=True)

In [ ]:
X = df.drop(["Pass_Fail","Student_ID"], axis=1)
Y = df["Pass_Fail"]

In [ ]:
numeric_cols =  X.select_dtypes("int").columns.tolist()
cat_cols =  X.select_dtypes("string").columns.tolist()

In [ ]:
lst = [x for x in cat_cols if x!="Pass_Fail" and x!="Student_ID"]

In [ ]:
xtrain, xtest, ytrain, ytest = train_test_split(X, Y, test_size = 0.2, random_state = 42)

In [ ]:
transformer = ColumnTransformer(transformers = [("Encoding", skp.OrdinalEncoder(), cat_cols)], remainder = "passthrough")

In [ ]:
pipe = Pipeline(steps=[("Transformation", transformer), ("Classification", DecisionTreeClassifier())])
pipe.fit(xtrain, ytrain)

In [ ]:
ypred = pipe.predict(xtest)

In [ ]:
print(f"Accuracy Score: {skm.accuracy_score(ytest, ypred)}")
print(f"Precision Score: {skm.precision_score(ytest, ypred, pos_label="Pass")}")
print(f"Recall Score: {skm.recall_score(ytest, ypred, pos_label="Pass")}")
print(f"F1 Score: {skm.f1_score(ytest, ypred, pos_label="Pass")}")

In [ ]:
for col in cat_cols:
    print(f"{col}: {df[col].unique()}")
    print(" ")

In [ ]:
print(f"Confusion Matrix:\n {skm.confusion_matrix(ytest, ypred)}")

In [ ]:
# 1. Prints a beautifully formatted text table of all main metrics at once
print(skm.classification_report(ytest, ypred))

In [ ]:
# Not Recommended
# with open("model","wb") as f:
#     pickle.dump(pipe, f)
    
# with open("model","rb") as f:
#     model = pickle.load(f)

In [ ]:
# Recommended
joblib.dump(pipe,"model")
model = joblib.load("model")

In [ ]:
model.predict(xtest)

In [ ]:
pd.DataFrame(model.predict(xtest),columns=["Result Column"])

In [ ]:
row3 =  X.sample(2)
row3

In [ ]:
# ColumnTransformer handles the extra value automatically given in prediction
model.predict(row3)

In [ ]:
user_data = {
    "Gender": "Female",
    "Study_Hours_per_Week": 20,
    "Attendance_Rate": 85.0,
    "Past_Exam_Scores": 75,
    "Parental_Education_Level": "Bachelors",
    "Internet_Access_at_Home": "Yes",
    "Extracurricular_Activities": "No",
    "Final_Exam_Score": 65
}

single_row_df = pd.DataFrame([user_data])

prediction = model.predict(single_row_df)

print("\n" + "="*40)
print(f"PREDICTION RESULT: The student is predicted to {prediction[0].upper()}")
print("="*40)
